# Deduplication of Job Postings Dataset

Removes reposts using a **rolling 60-day content-hash window**.
Deduplication runs across the *full combined dataset* (2021–2025) before splitting
by year, so cross-year reposts (e.g. a Dec posting re-listed in Jan) are caught correctly.

In [ ]:
import os
import hashlib
import pandas as pd

# ── Config (edit here) ────────────────────────────────────────────────────────
BASE_DIR       = r"../data/Postings"          # directory for 2021-2024 CSVs
DIR_2025       = r"../data/Postings/2025"     # directory for the 2025 CSV
FILE_2025      = os.path.join(DIR_2025, "2025_postings_merged_JAN_JUNE.csv")
COUNTRY_PREFIX = "SG"
WINDOW_DAYS    = 60
PARQUET_KWARGS = {"engine": "pyarrow", "compression": "snappy"}

# Columns used to build the content hash (missing cols are silently skipped)
HASH_COLS = [
    "ultimate_parent_rcid",
    "ultimate_parent_company_name",
    "company",
    "mapped_role",
    "onet_code",
    "jobtitle",
    "title_raw",
    "title_translated",
    "location_raw",
    "remote_type",
    "description",
    "salary_min",
    "salary_max",
    "salary_predicted",
]


In [ ]:
def _ensure_content_hash(
    df: pd.DataFrame,
    hash_cols: list | None = None,
    hash_colname: str = "content_hash",
) -> pd.DataFrame:
    """
    Add a stable MD5 content-hash column to *df* if one does not already exist.
    Existing non-null values are left untouched.
    """
    df = df.copy()

    if hash_colname in df.columns and df[hash_colname].notna().any():
        return df

    cols = [c for c in (hash_cols or HASH_COLS) if c in df.columns]
    if not cols:
        raise ValueError("No valid hash columns found in df.")

    def _norm(x):
        if pd.isna(x):
            return ""
        return " ".join(str(x).strip().lower().split())

    def _make_hash(row):
        return hashlib.md5("||".join(_norm(v) for v in row).encode("utf-8")).hexdigest()

    # raw=True passes a numpy array per row — much faster than axis=1 with Series
    df[hash_colname] = df[cols].apply(_make_hash, axis=1, raw=True)
    return df


def _dedupe_within_window(
    df: pd.DataFrame,
    date_col: str = "post_date",
    hash_col: str = "content_hash",
    window_days: int = WINDOW_DAYS,
    keep: str = "first",
) -> pd.DataFrame:
    """
    Rolling-window repost removal.

    For each content_hash:
      - sort chronologically
      - open a new 'episode' when the gap to the previous posting exceeds window_days
      - keep the first (or last) posting within each episode
    """
    if keep not in ("first", "last"):
        raise ValueError("keep must be 'first' or 'last'.")

    dd = df.sort_values([hash_col, date_col]).copy()

    prev_date   = dd.groupby(hash_col)[date_col].shift(1)
    gap_days    = (dd[date_col] - prev_date).dt.days
    new_episode = prev_date.isna() | (gap_days > window_days)

    # cumsum per hash group so episode IDs restart for each unique posting
    dd["_episode_id"] = new_episode.groupby(dd[hash_col]).cumsum()

    out = dd.drop_duplicates(subset=[hash_col, "_episode_id"], keep=keep)
    return out.drop(columns=["_episode_id"])


In [ ]:
print("Loading CSVs …")
frames = {}

for year in [2021, 2022, 2023, 2024]:
    path = os.path.join(BASE_DIR, f"{COUNTRY_PREFIX}-{year}.csv")
    frames[year] = pd.read_csv(path)
    print(f"  {year}: {len(frames[year]):,} rows")

frames[2025] = pd.read_csv(FILE_2025)
print(f"  2025: {len(frames[2025]):,} rows")

data_df = pd.concat(frames.values(), ignore_index=True)
print(f"\nCombined : {len(data_df):,} rows")


In [ ]:
# Parse dates — coerce bad values to NaT then drop them
data_df["post_date"] = pd.to_datetime(data_df["post_date"], errors="coerce")
n_bad_dates = data_df["post_date"].isna().sum()
data_df = data_df.dropna(subset=["post_date"]).reset_index(drop=True)
print(f"Dropped {n_bad_dates:,} rows with unparseable dates")
print(f"Remaining: {len(data_df):,} rows")

# Build content hash once over the full dataset.
# Doing this before the per-year split is essential: it ensures that two
# identical postings appearing in different years share the same hash, so the
# cross-year rolling window can eliminate the duplicate correctly.
print("\nBuilding content hashes …")
data_df = _ensure_content_hash(data_df, hash_colname="content_hash")
print("Done.")


In [ ]:
print(f"Running {WINDOW_DAYS}-day rolling-window deduplication on full dataset …")
data_deduped = _dedupe_within_window(
    data_df,
    date_col="post_date",
    hash_col="content_hash",
    window_days=WINDOW_DAYS,
    keep="first",
)
removed = len(data_df) - len(data_deduped)
print(f"Before : {len(data_df):,} rows")
print(f"After  : {len(data_deduped):,} rows  (removed {removed:,} reposts)")


In [ ]:
# Map each year to its output directory and desired filename
year_cfg = {
    year: {
        "out_dir": BASE_DIR,
        "fname": f"{COUNTRY_PREFIX}-{year}-WITHOUT-REPOSTS-W{WINDOW_DAYS}D.parquet",
    }
    for year in [2021, 2022, 2023, 2024]
}
year_cfg[2025] = {
    "out_dir": DIR_2025,
    "fname": f"2025_postings_merged_JAN_JUNE_WITHOUT_REPOSTS_W{WINDOW_DAYS}D.parquet",
}

print("Saving per-year parquet files …\n")
for year, cfg in year_cfg.items():
    os.makedirs(cfg["out_dir"], exist_ok=True)

    dy = data_deduped[data_deduped["post_date"].dt.year == year]
    out_path = os.path.join(cfg["out_dir"], cfg["fname"])
    dy.to_parquet(out_path, index=False, **PARQUET_KWARGS)

    raw = len(frames[year])
    print(f"[{year}]  raw: {raw:,}  →  deduped: {len(dy):,}  (removed {raw - len(dy):,})")
    print(f"         saved → {out_path}\n")
